# EDA — Componente A: Detección de Anomalías

Notebook exploratorio orientado al **diseño del autoencoder (AE)** para detección de campañas
agrícolas anómalas en la región núcleo (26 departamentos × soja/maíz, campañas 1981/82–2024/25).

Cada sección responde una pregunta concreta de diseño del AE. Principios respetados en todo el notebook:

- **Sin leakage del target**: `rinde_kgha` y sus derivados *nunca* son features del AE (solo se usan para etiquetar y como filtro de filas válidas).
- **Sin `dropna()` global**: el NDVI es NaN estructural pre-2002 (MODIS no existía). Los NaN se manejan localmente.
- **Estadísticas solo sobre train**: medias/desvíos de normalización se calculan con `campania_inicio ≤ 2017`.
- **Soja y maíz por separado**: escalas muy distintas (~2.700 vs ~6.700 kg/ha).

Splits temporales: **train** ≤ 2017 · **val** 2018–2020 · **test** 2021–2024 (incluye la sequía 2022/23).

## 1. Ground Truth Proxy — etiquetado formal de campañas anómalas

**Pregunta:** ¿qué campañas-departamento consideramos "anómalas" para validar el AE?
**Por qué importa:** el AE es no supervisado, pero necesitamos una etiqueta-proxy para medir si lo que
detecta como raro coincide con campañas realmente malas. Esta etiqueta *no* entra al AE como feature;
solo sirve de validación y para excluir campañas anómalas del entrenamiento (el AE aprende lo "normal").

**Criterio oficial** — una fila (campaña × depto × cultivo) es anómala si cumple **alguna**:
1. `z_rinde < -1.5`, con `z = (rinde - media_móvil_5) / std_móvil_5` (ventana hacia atrás, sin el año actual).
2. `oni_mean_octfeb <= -0.5` (La Niña).

## 2. Espacio de features del AE — correlación y redundancia

**Pregunta:** ¿cuál es la dimensionalidad real de las 49 variables NASA POWER que recibe el AE?
**Por qué importa:** define el tamaño del cuello de botella y si conviene de-correlacionar (las variables
mensuales de una misma magnitud están muy correlacionadas entre meses contiguos).

## 3. Campañas normales vs anómalas en el espacio climático

**Pregunta:** ¿las campañas anómalas se distinguen en las features que ve el AE?
**Por qué importa:** si normales y anómalas tienen distribuciones climáticas separables, el problema es
factible y el AE debería reconstruir mal las anómalas. Si no, ninguna arquitectura las detectará.

## 4. Proyecciones 2D — PCA, t-SNE y UMAP

**Pregunta:** ¿se separan visualmente normales y anómalas en 2D?, ¿val/test caen en zonas conocidas?
**Por qué importa:** una separación clara anticipa que el AE detectará bien; el shift de val/test indica
si el test exige extrapolación. Input: features NASA + `oni_mean_octfeb` normalizadas (sin rinde).

## 5. Variabilidad departamental de las features climáticas (train)

**Pregunta:** ¿la diferencia climática *entre* departamentos es comparable a la variabilidad *interanual*?
**Por qué importa:** si lo es, normalizar globalmente mezclaría la firma estructural de cada depto con la
anomalía de campaña. Justifica normalizar **por departamento** antes de alimentar el AE.

## 6. Implicancias para el diseño del autoencoder

| Hallazgo del EDA | Decisión de diseño |
|---|---|
| PCA: ~`n90` componentes explican el 90% de la varianza | Cuello de botella ≈ `n90//2`–`n90` neuronas |
| Correlación intra-variable alta (>0.9 entre meses contiguos) | De-correlacionar (PCA pre-AE) o regularizar; no esperar que el AE aprenda 49 dims independientes |
| Separabilidad visual en UMAP/t-SNE | Factibilidad del detector (ver conclusiones) |
| Features más discriminativas (Mann-Whitney): precip dic–feb, t2m_max ene, ONI | Deben tener buen error de reconstrucción en condiciones normales y dispararse en anómalas |
| Variación entre deptos ≈ interanual | Normalización **por departamento** (fit en train, transform en todo) |
| Campañas train con >30% deptos anómalos | Excluidas del entrenamiento del AE (ver lista impresa en sección 1) |
| Posición de val/test en UMAP | Indica si el test exige extrapolación (distribution shift) |

La celda siguiente exporta los artefactos que consumirá el notebook del autoencoder.

## Conclusiones para el modelado

> *(Los números entre paréntesis se calculan arriba; revisar las celdas si el panel cambió.)*

- **El etiquetado-proxy es coherente con la historia conocida**: las campañas con mayor % de departamentos anómalos coinciden con sequías documentadas (2022/23, 2008/09, La Niña 1988/89). El criterio ONI domina por su naturaleza global; el criterio z-rinde aporta las anomalías locales no climáticas.
- **El espacio climático es altamente redundante**: pares de meses contiguos de una misma variable superan r > 0.9 y PCA comprime el 90% de la varianza en pocos componentes → el cuello de botella del AE puede ser chico sin perder información.
- **Las features más discriminativas** entre normal y anómala (Mann-Whitney) son la **precipitación de dic–feb**, la **temperatura máxima de enero** y el **ONI** → el AE debe reconstruirlas bien en lo normal para que su error las delate en lo anómalo.
- **La sequía 2022/23 debería ser un outlier claro**: aparece desplazada en las proyecciones y con perfil mensual de lluvia muy por debajo de la media → el AE debería detectarla con alta confianza.
- **Heterogeneidad espacial fuerte**: la diferencia climática entre departamentos es comparable a la variabilidad interanual → **normalizar por departamento** (no global) para que el AE no confunda la firma de un depto seco con una anomalía de campaña.
- **La señal clima–rinde es heterogénea por depto**: algunos responden más a la lluvia, otros a la temperatura → un AE único sobre features normalizadas por depto es razonable, pero conviene revisar el error de reconstrucción por departamento.
- **Distribution shift en val/test**: revisar en UMAP si test (2021–2024) cae dentro de la nube de train; de caer en zona de extrapolación, las anomalías detectadas allí deben interpretarse con cautela.
- **Entrenar el AE solo con lo normal**: excluir del fit las filas anómalas y las campañas train con >30% de deptos anómalos, para que el AE aprenda la distribución de campañas sanas.